# Build Your Own Deep Research Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jaskamalkainth/Multi-Agent-Patterns/blob/main/notebooks/07_deep_research_crewai.ipynb)

Have you used *Deep Research* or *Research* mode in Claude, ChatGPT, Gemini or Perplexity? In Claude's case it
isn't one AI working alone: it's a **team of agents**. A lead plans the research, several researchers work on
it **at the same time**, and the results come together in one report
([how Anthropic built it](https://www.anthropic.com/engineering/multi-agent-research-system)).

![A lead researcher plans and sends three subagents to search in parallel; the lead reviews their notes; a citation agent ties claims to sources](https://raw.githubusercontent.com/Jaskamalkainth/Multi-Agent-Patterns/main/notebooks/images/deep-research-architecture.svg)

In this notebook you build a small version with [CrewAI](https://docs.crewai.com):

![Your crew: your topic goes to a Lead Researcher who writes a plan; three researchers work on one sub-question each, all at once; a Report Writer waits for all three and writes the report](https://raw.githubusercontent.com/Jaskamalkainth/Multi-Agent-Patterns/main/notebooks/images/07-crew.svg)

### How to use this notebook

1. **Get a free API key.** [Groq](https://console.groq.com/keys) or [Gemini](https://aistudio.google.com/api-keys)
   both have a free tier. (Or use `ollama` on your own laptop, with no key at all.)
2. **Make the key available**, whichever is easiest:
   - **In Colab:** click the 🔑 **Secrets** icon in the left sidebar → **Add new secret** → name it `GROQ_API_KEY`
     (or `GEMINI_API_KEY`) → paste the key → turn on **Notebook access**.
   - **On your laptop:** put `GROQ_API_KEY=...` in the repo's `.env` file. CrewAI needs Python 3.10–3.13.
   - **Neither?** Step ② asks you to paste the key, and it stays hidden.
3. **Run the cells in order** with ▶ (in Colab: **Runtime → Run all**). The ①② setup cells are short: in Colab
   they show as forms with the code hidden.
4. **Change the settings** (your topic, the team size, parallel or not): in Colab they're form fields on the
   right of a cell; elsewhere, edit the value on the first line of the cell. Then run that cell and the ones
   below it again.

In [ ]:
# @title ① Pick your LLM provider and install CrewAI (about a minute)
PROVIDER = "groq"  # @param ["groq", "gemini", "openai", "anthropic", "openrouter", "ollama"]
# @markdown `groq` and `gemini` have free keys. `ollama` runs on your own laptop (not in Colab), with no key.
# @markdown If Colab asks you to restart the session after the install, click **Restart**, then run this cell
# @markdown once more and carry on.

EXTRAS = {"gemini": "tools,google-genai", "anthropic": "tools,anthropic"}.get(PROVIDER, "tools")
%pip install -q "crewai[{EXTRAS}]==1.15.28"
print("✅ CrewAI installed for", PROVIDER)

In [ ]:
# @title ② Connect your API key
import os
from getpass import getpass
from crewai import LLM
from dotenv import load_dotenv

load_dotenv("../.env")   # on your laptop: keys from the repo's .env file (in Colab there is none)

KEY_NAMES = {"groq": "GROQ_API_KEY", "gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY",
             "anthropic": "ANTHROPIC_API_KEY", "openrouter": "OPENROUTER_API_KEY"}
MODELS = {
    "groq":       "openai/openai/gpt-oss-20b",   # CrewAI's OpenAI client + Groq's id "openai/gpt-oss-20b"
    "gemini":     "gemini/gemini-3.5-flash",
    "openai":     "openai/gpt-4o-mini",
    "anthropic":  "anthropic/claude-haiku-5-5",
    "openrouter": "openrouter/meta-llama/llama-3.3-70b-instruct",
    "ollama":     "ollama/llama3.2",               # no key: install Ollama, then `ollama pull llama3.2`
}

key_name = KEY_NAMES.get(PROVIDER)   # ollama needs no key
if key_name and not os.getenv(key_name):
    try:
        from google.colab import userdata
        os.environ[key_name] = userdata.get(key_name)
        print(f"🔑 Using {key_name} from Colab Secrets")
    except Exception:   # not in Colab, no secret with that name, or no notebook access
        os.environ[key_name] = getpass(f"Paste your {key_name} (it stays hidden): ")

settings = {} if PROVIDER == "anthropic" else {"temperature": 0.3}   # Claude rejects temperature
if PROVIDER == "groq":       # Groq speaks the OpenAI API; its free tier answers 429 when busy, so retry
    settings |= {"base_url": "https://api.groq.com/openai/v1", "api_key": os.environ[key_name],
                 "custom_openai": True, "max_retries": 6}
if PROVIDER == "gemini":     # Gemini's free tier often answers 503 "high demand": retry with backoff
    settings["client_params"] = {"http_options": {"retry_options": {"attempts": 6, "initial_delay": 5}}}

llm = LLM(model=MODELS[PROVIDER], **settings)
print("✅", MODELS[PROVIDER], "says:", llm.call("Reply with just: OK"))

## ③ The agents

CrewAI has three building blocks, and this is all of CrewAI you need:

![CrewAI's three building blocks: an Agent (role, goal, backstory) is who does the work, a Task (description, expected_output, agent) is what to do, and a Crew (agents, tasks, kickoff) is the team](https://raw.githubusercontent.com/Jaskamalkainth/Multi-Agent-Patterns/main/notebooks/images/07-building-blocks.svg)

Your crew has a lead, a few researchers and a writer. The same LLM plays all of them: the `role`, `goal` and
`backstory` are what make them behave differently. The researchers are made in a loop, one per sub-question,
because CrewAI runs one agent on one task at a time.

In [ ]:
from crewai import Agent, Task, Crew

TEAM_SIZE = 3  # @param {type:"slider", min:1, max:5, step:1}

lead = Agent(
    role="Lead Researcher",
    goal="Split a research question into clear sub-questions that don't overlap",
    backstory="You plan research for a small team. You never answer the question yourself: "
              "you decide who looks into what.",
    llm=llm,
)

researchers = [
    Agent(
        role=f"Researcher {n}",
        goal=f"Find the key facts for sub-question {n} of the lead's plan",
        backstory="You research exactly one sub-question from the lead's plan, and only that one. "
                  "Other researchers cover the rest. You say when you're unsure.",
        llm=llm,
    )
    for n in range(1, TEAM_SIZE + 1)
]

writer = Agent(
    role="Report Writer",
    goal="Turn the team's findings into a short, clear report",
    backstory="You write for busy readers: the answer first, then the evidence. "
              "You only use the findings the researchers give you.",
    llm=llm,
)

## ④ The tasks: plan → research in parallel → report

- **`{topic}`** and **`{team_size}`** are filled in when the crew runs, so the same team researches any question.
- **`context=[plan]`** hands the plan to each researcher. **`context=research`** hands every researcher's findings
  to the writer.
- **`async_execution=True`** makes the research tasks start together instead of one after another. The report
  waits until all of them are done.

![Timeline: in parallel, the researchers run at the same time and the report is ready sooner; one after another, the same work takes much longer](https://raw.githubusercontent.com/Jaskamalkainth/Multi-Agent-Patterns/main/notebooks/images/07-parallel.svg)

In [ ]:
PARALLEL = True  # @param {type:"boolean"}

plan = Task(
    description="Split this research question into exactly {team_size} sub-questions that don't overlap: {topic}",
    expected_output="{team_size} numbered sub-questions, one line each",
    agent=lead,
)

research = [
    Task(
        description=f"Research sub-question {n} from the plan. Cover only sub-question {n}.",
        expected_output="5 bullet points, one short fact each",
        agent=researchers[n - 1],
        context=[plan],
        async_execution=PARALLEL,   # True: all researchers at the same time
    )
    for n in range(1, TEAM_SIZE + 1)
]

report = Task(
    description="Write a short report that answers: {topic}",
    expected_output="Markdown: a one-sentence answer, then one short section per sub-question. At most 300 words.",
    agent=writer,
    context=research,   # waits for every researcher
)

crew = Crew(agents=[lead, *researchers, writer], tasks=[plan, *research, report])
print(f"✅ Crew ready: a lead, a writer and {TEAM_SIZE} researcher(s) · parallel = {PARALLEL}")

In [ ]:
# @title Show a one-line trace while the crew works
from crewai.events import crewai_event_bus, AgentExecutionStartedEvent, TaskCompletedEvent

if "TRACE_ON" not in globals():   # register once, even if you run this cell again
    TRACE_ON = True

    @crewai_event_bus.on(AgentExecutionStartedEvent)
    def on_agent_started(source, event):
        print(f"▶ {event.agent.role} started")

    @crewai_event_bus.on(TaskCompletedEvent)
    def on_task_completed(source, event):
        print(f"✔ {event.output.agent}: {event.output.raw[:100]!r}…\n")

print("✅ Trace on: watch for the researchers starting one right after the other")

## ⑤ Run your research

Type your question in the `topic` field and run the cell. Questions with a few separate parts work best:
comparisons, pros and cons, "how did X change over time".

In [ ]:
topic = "How do electric cars compare with petrol cars on cost, environment and convenience?"  # @param {type:"string"}

import time
from IPython.display import Markdown, display

start = time.time()
result = await crew.kickoff_async(inputs={"topic": topic, "team_size": TEAM_SIZE})
print(f"\n⏱ done in {time.time() - start:.0f} seconds")
display(Markdown(result.raw))

In [ ]:
# @title Look inside: every agent's output, and what the run cost
for t in result.tasks_output:
    print(f"── {t.agent} ──\n{t.raw}\n")
print(result.token_usage)

## ⑥ Build your own

Change a setting, then run that cell and every cell below it again (**Runtime → Run cell and below**).

1. **Your question.** Change `topic` in ⑤.
2. **A bigger team.** Move the `TEAM_SIZE` slider in ③ to 5. More facts? Look at the token count.
3. **One at a time.** Untick `PARALLEL` in ④ and compare the seconds. Same work, same tokens: only the
   waiting changes.
4. **Add a skeptic.** In ③, add an agent whose backstory says to look for evidence *against* the obvious
   answer. In ④, give it a task with `context=[plan]` and `async_execution=PARALLEL`, add the task to `research`,
   and add the agent to the crew. Does the report become more balanced?
5. **Change a personality.** Edit the writer's `backstory`, for example "You explain things to a
   10-year-old". Same research, different report.
6. **Give the researchers web search** (needs a free key from [serper.dev](https://serper.dev)). Run this
   before ③, then rerun ③ onwards:
   ```python
   from crewai_tools import SerperDevTool
   os.environ["SERPER_API_KEY"] = getpass("SERPER_API_KEY: ")
   search = SerperDevTool()
   ```
   and add `tools=[search]` to each researcher `Agent(...)`.

## Key notes

| Piece | Pattern | In Claude's Research |
|---|---|---|
| `plan` by the Lead Researcher | Hierarchical: a lead splits the work | The lead agent writes a plan and starts subagents |
| Research tasks with `async_execution=True` | Parallel fan-out | 3–5 subagents work at the same time |
| `context=research` on the report | Fan-in: wait for everyone, then combine | The lead collects the findings |
| Each researcher sees only the plan and its own task | Selective context | Each subagent has its own context window |

**Worth remembering** (from [Anthropic's write-up](https://www.anthropic.com/engineering/multi-agent-research-system))

- A multi-agent system (Claude Opus 4 lead, Claude Sonnet 4 subagents) beat a single Claude Opus 4 agent by
  **90.2%** on Anthropic's internal research eval.
- It costs more: about **15×** the tokens of a chat (one agent: about 4×).
- Running subagents in parallel cut research time by **up to 90%** on complex questions.
- Teams fit questions that split into **independent parts**. When every step depends on the one before, one
  agent is usually better.

**Best practices**

1. **One job per agent**, with a specific role.
2. **Always write `expected_output`**, with a format and a length.
3. **Run independent work in parallel** with `async_execution=True`, and fan back in with `context`.
4. **Read the intermediate outputs**, not just the report: that's how you find which agent to fix.
5. **Start simple.** Add agents only when they make the answer better, and check the token cost.

## Sources

- Anthropic, [How we built our multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system) (June 2025)
- Anthropic, [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) (December 2024)
- CrewAI docs: [Tasks](https://docs.crewai.com/en/concepts/tasks) (`context`, `async_execution`) and [Crews](https://docs.crewai.com/en/concepts/crews)
- The same crew in LangGraph: `06_deep_research_langgraph.ipynb`, with a review loop, charts and a quiz
- The full repo, with eight coordination patterns animated and more notebooks:
  [github.com/Jaskamalkainth/Multi-Agent-Patterns](https://github.com/Jaskamalkainth/Multi-Agent-Patterns)